In [1]:
import os
from pathlib import Path
from copy import deepcopy

import dotenv
import pandas as pd
import pyam
from pandas_openscm.io import load_timeseries_csv
import silicone.database_crunchers

import gcages
from gcages import renaming
from gcages.cmip7_scenariomip.harmonisation import create_cmip7_scenariomip_global_harmoniser
from gcages.cmip7_scenariomip.infilling import CMIP7ScenarioMIPInfiller

dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
[WARNING] 23:24:15 - pint.util: Redefining 'C' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:15 - pint.util: Redefining 'N' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:15 - pint.util: Redefining 'NOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:15 - pint.util: Redefining 'gNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:15 - pint.util: Redefining 'tNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:15 - pint.util: Redefining 'S' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 23:24:16 - pint.util: Redefining 'yr' (<class 'p

# Configuration and File Paths

In [7]:
# Input and output paths
INPUT_DATA_FILE = OUTPUT_FOLDER / "100_inputdata_20260522.csv"
INTERMEDIATE_OUTPUT = OUTPUT_FOLDER / "intermediate/101_intermediate_infilling_db.csv"
INTERMEDIATE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# CMIP7 configuration paths
BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"
ANERIS_GLOBAL_OVERRIDES = BASE_DIR / "aneris-overrides-global.csv"
CMIP7_INFILLING_FILE = BASE_DIR / "infilling_db_cmip7_scenariomip.csv"
GHG_INVERSION_FILE = BASE_DIR / "cmip7_ghg_inversions.csv"

In [8]:
# Database cruncher configuration for infilling missing variables
VARS_DB_CRUNCHERS = {
    "Emissions|BC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CH4": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO2|Biosphere": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO2|Fossil": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|N2O": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NH3": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NOx": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|OC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|SOx": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NMVOC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|HFC134a": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC143a": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC227ea": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC23": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC32": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC4310mee": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC125": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|SF6": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|CF4": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|C2F6": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|C6F14": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|CCl4": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC11": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC113": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC114": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC115": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC12": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH2Cl2": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3Br": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3CCl3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3Cl": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CHCl3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC141b": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC142b": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC22": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC152a": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC236fa": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC365mfc": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1202": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1211": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1301": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon2402": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|NF3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C3F8": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C4F10": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C5F12": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C7F16": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C8F18": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|cC4F8": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|SO2F2": (True, silicone.database_crunchers.RMSClosest),
}

LEAD_VARIABLES = ["Emissions|CO2"]

## Step 1: Load and Prepare Input Data

In [9]:
scenario_data = load_timeseries_csv(
    INPUT_DATA_FILE,
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [10]:
scenario_data_renamed = renaming.rename_variables(
    scenario_data,
    from_convention=renaming.SupportedNamingConventions.CMIP7_SCENARIOMIP,
    to_convention=renaming.SupportedNamingConventions.GCAGES
)

# Fix HFC43-10 unit naming (remove hyphen)
scenario_data_renamed = scenario_data_renamed.reset_index()
scenario_data_renamed['unit'] = scenario_data_renamed['unit'].replace(
    'kt HFC43-10/yr', 'kt HFC4310/yr'
)
scenario_data_renamed.set_index(
    ["model", "scenario", "region", "variable", "unit"], 
    inplace=True
)

## Step 2: Harmonize Scenario Data to Historical Emissions

In [11]:
harmoniser = create_cmip7_scenariomip_global_harmoniser(
    CMIP7_HISTORICAL_EMISSIONS,
    ANERIS_GLOBAL_OVERRIDES,
    run_checks=False
)

harmonised_data = harmoniser(scenario_data_renamed)

FileNotFoundError: [Errno 2] No such file or directory: '../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/history_cmip7_scenariomip.csv'

## Step 3: Prepare for Infilling - Create Intermediate Database

In [7]:
harmonised_pyam = pyam.IamDataFrame(harmonised_data)

# Add total CO2 emissions (Fossil + Biosphere)
harmonised_pyam.add(
    a='Emissions|CO2|Fossil',
    b='Emissions|CO2|Biosphere',
    name='Emissions|CO2',
    ignore_units='Mt CO2/yr',
    append=True
)

# Save intermediate database for infilling
harmonised_pyam.timeseries().to_csv(INTERMEDIATE_OUTPUT)

## Step 4: Infill Missing Variables Using AR6-Style Approach

In [8]:
def infill_missing_variable(variable, harmonised_pyam_data, cruncher_class, lead_vars):
    """Infill a single missing variable using specified database cruncher."""
    df_copy = deepcopy(harmonised_pyam_data)
    df_copy.require_data(variable=variable, exclude_on_fail=True)
    data_to_infill = df_copy.filter(exclude=True)
    
    if data_to_infill.empty:
        return None
    
    print(f"Infilling {variable}")
    cruncher = cruncher_class(harmonised_pyam_data)
    filler = cruncher.derive_relationship(variable, lead_vars)
    return filler(data_to_infill)


# Identify missing variables from CMIP7 infiller
cmip7_infiller = CMIP7ScenarioMIPInfiller.from_cmip7_scenariomip_config(
    cmip7_scenariomip_infilling_leader_emissions_file=CMIP7_INFILLING_FILE,
    cmip7_ghg_inversions_file=GHG_INVERSION_FILE,
    cmip7_scenariomip_global_historical_emissions_file=CMIP7_HISTORICAL_EMISSIONS,
)

available_vars = set(cmip7_infiller.infilling_db.index.get_level_values('variable').unique())
ghg_inversion_vars = set(cmip7_infiller.cmip7_ghg_inversions.index.get_level_values('variable'))
cmip7_vars = available_vars.union(ghg_inversion_vars)

from gcages.cmip7_scenariomip.infilling import COMPLETE_EMISSIONS_INPUT_VARIABLES_GCAGES
missing_variables = list(set(COMPLETE_EMISSIONS_INPUT_VARIABLES_GCAGES).difference(cmip7_vars))

In [9]:
missing_variables

['Emissions|CO2|Biosphere',
 'Emissions|NH3',
 'Emissions|NMVOC',
 'Emissions|SOx',
 'Emissions|OC',
 'Emissions|NOx',
 'Emissions|CO',
 'Emissions|CH4',
 'Emissions|BC']

In [10]:
# Infill all missing variables
infilled_dataframes = []
for variable in missing_variables:
    if variable not in VARS_DB_CRUNCHERS:
        continue
    
    _, cruncher_class = VARS_DB_CRUNCHERS[variable]
    
    try:
        infilled = infill_missing_variable(
            variable, 
            harmonised_pyam, 
            cruncher_class, 
            LEAD_VARIABLES
        )
        if infilled is not None:
            infilled_dataframes.append(infilled)
    except Exception as e:
        print(f"Failed to infill {variable}: {e}")

[WARNING] 11:23:22 - pyam.core: Filtered IamDataFrame is empty!
[INFO] 11:23:22 - pyam.validation: 357 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|NH3


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 11:23:23 - pyam.validation: 357 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|NMVOC


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 11:23:24 - pyam.validation: 126 scenarios failed validation and will be set as `exclude=True`.


...

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 11:23:28 - pyam.validation: 6 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|CH4


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 11:23:28 - pyam.validation: 364 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|BC


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)


## Step 5: Combine Infilled Data with Harmonised Data

In [11]:
intermediate_infilled = pyam.concat(infilled_dataframes)
intermediate_infilled = pyam.concat([intermediate_infilled, harmonised_pyam])

# Remove total CO2 (will be recalculated by CMIP7 infiller)
intermediate_infilled.filter(variable='Emissions|CO2', keep=False, inplace=True)

# Remove problematic scenario
intermediate_infilled.filter(
    model='COFFEE 1.5',
    scenario='NAVIGATE*',
    keep=False,
    inplace=True
)

intermediate_timeseries = intermediate_infilled.timeseries()

## Step 6: Complete Infilling with CMIP7 ScenarioMIP Infiller

In [12]:
complete_data = cmip7_infiller(intermediate_timeseries)
complete_data

year                                                                                      2023  \
model             scenario              region variable                unit                      
AIM/CGE 2.0       SSP1-19               World  Emissions|BC            Mt BC/yr       7.219245   
                                               Emissions|CH4           Mt CH4/yr    363.489901   
                                               Emissions|CO            Mt CO/yr     774.053434   
                                               Emissions|CO2|Biosphere Mt CO2/yr   3627.550667   
                                               Emissions|CO2|Fossil    Mt CO2/yr  38712.217040   
...                                                                                        ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi          World  Emissions|NF3           kt NF3/yr      4.010291   
                  CD-LINKS-NPi2020_1000 World  Emissions|NF3           kt NF3/yr      4.010291   
                  CD-LINKS-NPi2020_1600 World  Emissions|NF3           kt NF3/yr      4.010291   
                  CD-LINKS-NPi2020_400  World  Emissions|NF3           kt NF3/yr      4.010291   
                  CD-LINKS-No-Policy    World  Emissions|NF3           kt NF3/yr      4.010291   

year                                                                                      2024  \
model             scenario              region variable                unit                      
AIM/CGE 2.0       SSP1-19               World  Emissions|BC            Mt BC/yr       6.942491   
                                               Emissions|CH4           Mt CH4/yr    332.845294   
                                               Emissions|CO            Mt CO/yr     758.003015   
                                               Emissions|CO2|Biosphere Mt CO2/yr   3249.784994   
                                               Emissions|CO2|Fossil    Mt CO2/yr  36639.056595   
...                                                                                        ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi          World  Emissions|NF3           kt NF3/yr      4.112262   
                  CD-LINKS-NPi2020_1000 World  Emissions|NF3           kt NF3/yr      4.110169   
                  CD-LINKS-NPi2020_1600 World  Emissions|NF3           kt NF3/yr      4.013497   
                  CD-LINKS-NPi2020_400  World  Emissions|NF3           kt NF3/yr      3.833696   
                  CD-LINKS-No-Policy    World  Emissions|NF3           kt NF3/yr      4.112262   

year                                                                                      2025  \
model             scenario              region variable                unit                      
AIM/CGE 2.0       SSP1-19               World  Emissions|BC            Mt BC/yr       6.667881   
                                               Emissions|CH4           Mt CH4/yr    302.379832   
                                               Emissions|CO            Mt CO/yr     741.987254   
                                               Emissions|CO2|Biosphere Mt CO2/yr   2872.019322   
                                               Emissions|CO2|Fossil    Mt CO2/yr  34582.284727   
...                                                                                        ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi          World  Emissions|NF3           kt NF3/yr      4.214376   
                  CD-LINKS-NPi2020_1000 World  Emissions|NF3           kt NF3/yr      4.210176   
                  CD-LINKS-NPi2020_1600 World  Emissions|NF3           kt NF3/yr      4.016588   
                  CD-LINKS-NPi2020_400  World  Emissions|NF3           kt NF3/yr      3.659389   
                  CD-LINKS-No-Policy    World  Emissions|NF3           kt NF3/yr      4.214376   

year                                                                                      2026  \
model             scenario              region variable                unit  

In [13]:
complete_data.to_csv(OUTPUT_FOLDER / "101_harmonised_data.csv")